In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import joblib

In [2]:
# Cell 2: Load Data + Check Exact Column Names
df = pd.read_csv('/content/brisbane_water_quality.csv')

print("Columns in the file:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head(3))
df.head()

Columns in the file:
['Timestamp', 'Record number', 'Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']

First few rows:
             Timestamp  Record number  Average Water Speed  \
0  2023-08-04 23:00:00           1468                4.834   
1  2023-08-04 23:30:00           1469                2.544   
2  2023-08-04 23:00:00           1470                1.260   

   Average Water Direction  Chlorophyll  Chlorophyll [quality]  Temperature  \
0                   73.484        1.621                    NaN       20.018   
1                  106.424        1.959                    NaN       19.986   
2                  156.755        1.6

,Timestamp,Record number,Average Water Speed,Average Water Direction,Chlorophyll,Chlorophyll [quality],Temperature,Temperature [quality],Dissolved Oxygen,Dissolved Oxygen [quality],Dissolved Oxygen (%Saturation),Dissolved Oxygen (%Saturation) [quality],pH,pH [quality],Salinity,Salinity [quality],Specific Conductance,Specific Conductance [quality],Turbidity,Turbidity [quality]
0,2023-08-04 23:00:00,1468,4.834,73.484,1.621,NaN,20.018,NaN,7.472,NaN,101.175,NaN,8.176,NaN,35.215,NaN,53.262,NaN,2.068,NaN
1,2023-08-04 23:30:00,1469,2.544,106.424,1.959,NaN,19.986,NaN,7.455,NaN,100.884,NaN,8.175,NaN,35.209,NaN,53.254,NaN,1.994,NaN
2,2023-08-04 23:00:00,1470,1.260,156.755,1.620,NaN,20.001,NaN,7.430,NaN,100.571,NaN,8.171,NaN,35.207,NaN,53.252,NaN,2.030,NaN
3,2023-08-04 23:30:00,1471,0.760,281.754,1.761,NaN,19.983,NaN,7.419,NaN,100.398,NaN,8.171,NaN,35.211,NaN,53.257,NaN,1.973,NaN
4,2023-08-04 23:00:00,1472,3.397,244.637,1.635,NaN,19.986,NaN,7.429,NaN,100.538,NaN,8.171,NaN,35.208,NaN,53.253,NaN,1.944,NaN


In [3]:
print("Columns:")
print(df.columns)

Columns:
Index(['Timestamp', 'Record number', 'Average Water Speed',
       'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]',
       'Temperature', 'Temperature [quality]', 'Dissolved Oxygen',
       'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)',
       'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]',
       'Salinity', 'Salinity [quality]', 'Specific Conductance',
       'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]'],
      dtype='object')


In [4]:
features = [
    "pH",
    "Dissolved Oxygen",
    "Temperature",
    "Salinity",
    "Turbidity",
    "Chlorophyll"
]

X = df[features]
X.head()

,pH,Dissolved Oxygen,Temperature,Salinity,Turbidity,Chlorophyll
0,8.176,7.472,20.018,35.215,2.068,1.621
1,8.175,7.455,19.986,35.209,1.994,1.959
2,8.171,7.430,20.001,35.207,2.030,1.620
3,8.171,7.419,19.983,35.211,1.973,1.761
4,8.171,7.429,19.986,35.208,1.944,1.635


In [5]:
X = X.dropna()

print("After cleaning:", X.shape)

After cleaning: (20035, 6)


In [6]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaling done")

Scaling done


In [7]:
model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)

print("Model trained")

Model trained


In [8]:
pred = model.predict(X_scaled)

# Add to dataframe
X["Anomaly"] = pred

# Count
print(X["Anomaly"].value_counts())

Anomaly
 1    19033
-1     1002
Name: count, dtype: int64


In [9]:
anomalies = X[X["Anomaly"] == -1]

print("Number of anomalies:", len(anomalies))
anomalies.head()

Number of anomalies: 1002


,pH,Dissolved Oxygen,Temperature,Salinity,Turbidity,Chlorophyll,Anomaly
15,8.166,4.890,19.652,35.272,3.375,1.421,-1
61,8.163,4.837,19.860,35.255,1.632,1.258,-1
395,8.171,5.298,20.166,35.194,1.080,1.672,-1
1598,8.073,7.533,22.451,35.218,1.880,9.227,-1
10057,7.941,6.229,27.137,32.373,1.902,9.865,-1


In [10]:
joblib.dump(model, "if_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("✅ Model and scaler saved")

✅ Model and scaler saved


In [11]:
# Example NORMAL input (from your dataset)
input_data = np.array([[8.17, 7.45, 20.0, 35.2, 2.0, 1.6]])

# Scale it
input_scaled = scaler.transform(input_data)

# Predict
prediction = model.predict(input_scaled)

print("Prediction:", prediction)

if prediction[0] == 1:
    print("✅ NORMAL")
else:
    print("🚨 ANOMALY")

Prediction: [1]
✅ NORMAL


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [12]:
# Extreme values
input_data = np.array([[5, 2, 40, 10, 50, 30]])

input_scaled = scaler.transform(input_data)
prediction = model.predict(input_scaled)

print("Prediction:", prediction)

if prediction[0] == 1:
    print("✅ NORMAL")
else:
    print("🚨 ANOMALY")

Prediction: [-1]
🚨 ANOMALY


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
